![Project](https://img.shields.io/badge/NS%20Health%20%26%20Population%20Analytics-1B3A6B?style=for-the-badge&logoColor=white)

# ⏳ Life Expectancy — Data Cleaning Pipeline
![Python](https://img.shields.io/badge/Python_3.12-C9A020?style=flat-square&logo=python&logoColor=white) ![pandas](https://img.shields.io/badge/pandas-1A6B5A?style=flat-square&logo=pandas&logoColor=white) ![Source](https://img.shields.io/badge/Source-Statistics_Canada-1B3A6B?style=flat-square)

> **Used in:** Health Outcome Index (HOI) scatter plot · Executive Summary KPI card

Life expectancy is a foundational health outcome measure. This notebook cleans the raw StatCan life expectancy life tables for Nova Scotia and reshapes them into a tidy long-format dataset.

**Critical design note:** In the scatter plot (Income vs. Health Outcome Index), life expectancy is **inverted** after normalization — so that higher values on the Y-axis represent *worse* health outcomes, making the income-health correlation read intuitively.

---

## 📦 Source Data

| File | Gender | Coverage |
|------|--------|----------|
| `Life Expectancy Males.csv` | M | NS, by age group and period |
| `Life Expectancy Females.csv` | F | NS, by age group and period |

**Source:** Statistics Canada — *Complete life tables, Canada, provinces and territories*

---

## 🔧 Cleaning Steps Overview

| Step | Action |
|------|--------|
| 1 | Imports & file paths |
| 2 | Smart StatCan boundary reader (find `Age group` → stop at `Footnotes`) |
| 3 | Drop non-data rows, rename columns |
| 4 | Unpivot wide → long (`pd.melt`) |
| 5 | Strip quote artifacts, cast to numeric |
| 6 | Tag gender, combine M + F, export |

---


## Step 1 — Imports & File Paths


In [ ]:
import pandas as pd
from io import StringIO

MALE_FILE   = r"./data/Life Expectancy Males.csv"
FEMALE_FILE = r"./data/Life Expectancy Females.csv"
OUTPUT_FILE = r"./clean/life_expectancy_clean.csv"


## Step 2 — Smart StatCan Life Table Reader

Life expectancy CSVs share the same metadata-heavy structure as population files.
We scan line-by-line and extract only the rows between the `Age group` header and the `Footnotes:` section.

This approach is **robust to future StatCan formatting changes** — it works by content detection, not row position.


In [ ]:
def read_statscan_life_table(file_path):
    """
    Reads a StatCan life expectancy CSV safely by:
    - Detecting the real data table via content scanning
    - Skipping metadata rows above 'Age group' header
    - Stopping at 'Footnotes:' section
    - Parsing only the clean data slice
    """
    with open(file_path, "r", encoding="utf-8-sig", errors="ignore") as f:
        lines = f.readlines()

    start_idx = None
    end_idx   = None

    for i, line in enumerate(lines):
        if start_idx is None and "Age group" in line:
            start_idx = i
        elif start_idx is not None and "Footnotes:" in line:
            end_idx = i
            break

    if start_idx is None:
        raise ValueError(f"Could not locate life expectancy table in: {file_path}")

    if end_idx is None:
        end_idx = len(lines)

    table_text = "".join(lines[start_idx:end_idx])
    return pd.read_csv(StringIO(table_text), engine="python")


## Step 3 — Clean & Reshape

The life table is in **wide format** — age groups as rows, time periods (e.g. `2010-2012`, `2011-2013`) as columns.

We:
1. Rename the first column to `Age group` (StatCan adds footnote superscripts)
2. Drop the `Number` row (a unit label, not data)
3. Melt wide → long so each row is one (Age group × Period) observation
4. Strip any embedded quote characters from numeric values
5. Cast `Life Expectancy` to float
6. Tag with gender label


In [ ]:
def process_life_expectancy(file_path, gender_label):
    """
    Full cleaning pipeline for one StatCan life expectancy CSV.

    Returns long-format DataFrame:
      Age group | Year Period | Life Expectancy | Gender
    """
    df = read_statscan_life_table(file_path)

    # ── Rename first column (may have footnote superscripts) ─
    df.rename(columns={df.columns[0]: "Age group"}, inplace=True)

    # ── Drop non-data rows ───────────────────────────────────
    df = df[df["Age group"].notna()]
    df = df[~df["Age group"].isin(["Number"])]

    # ── Unpivot: wide → long ─────────────────────────────────
    # Year Period becomes a column e.g. '2010-2012', '2011-2013'
    df_long = df.melt(
        id_vars="Age group",
        var_name="Year Period",
        value_name="Life Expectancy"
    )

    # ── Strip embedded quote artifacts from numeric values ───
    df_long["Life Expectancy"] = (
        df_long["Life Expectancy"]
        .astype(str)
        .str.replace('"', "", regex=False)
    )

    # ── Cast to float (errors='coerce' returns NaN on failure) ─
    df_long["Life Expectancy"] = pd.to_numeric(
        df_long["Life Expectancy"], errors="coerce"
    )

    # ── Drop rows where life expectancy could not be parsed ──
    df_long = df_long.dropna(subset=["Life Expectancy"])

    # ── Tag gender ──────────────────────────────────────────
    df_long["Gender"] = gender_label

    return df_long[["Age group", "Year Period", "Life Expectancy", "Gender"]]


## Step 4 — Run Pipeline & Export


In [ ]:
male_df   = process_life_expectancy(MALE_FILE,   "M")
female_df = process_life_expectancy(FEMALE_FILE, "F")

final_df = pd.concat([male_df, female_df], ignore_index=True)
final_df = final_df.sort_values(by=["Age group", "Gender", "Year Period"])

import os
os.makedirs("./clean", exist_ok=True)
final_df.to_csv(OUTPUT_FILE, index=False)

print("✅ Life expectancy dataset cleaned successfully")
print("✅ Output saved to:", OUTPUT_FILE)
print("✅ Final shape:    ", final_df.shape)
print()
print(final_df.head(10))


## ✅ Output Summary

| Output | Location | Shape |
|--------|----------|-------|
| Clean CSV | `./clean/life_expectancy_clean.csv` | ~2,886 rows × 4 cols |

**Final schema:** `Age group` \| `Year Period` \| `Life Expectancy` \| `Gender`

**Feeds into:**
- 📊 Power BI — Executive Summary median life expectancy KPI
- 🧮 `Health_Outcome_Index.ipynb` — normalized & inverted for scatter plot Y-axis

> ⚠️ **Inversion note:** When building the HOI, life expectancy is transformed as:
> `life_expectancy_inv = (max_LE - LE) / (max_LE - min_LE)`
> This ensures higher index values = worse outcomes, so income correlates positively with the index.


---

*Nova Scotia Health & Population Analytics · Statistics Canada Life Tables*
